---
# 🛡️ Project 1 — SchemaShield: Structured Output Agent
---

Install Dependencies

In [1]:
!pip install -q pydantic langchain langchain-groq langchain-core tenacity loguru

API Key Setup

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

Imports

In [3]:
import os
import json
import re
from datetime import datetime, timezone
from typing import Literal
from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from tenacity import retry, stop_after_attempt, wait_exponential
from loguru import logger
import sys

logger.remove()
logger.add(
    sys.stdout,
    format="{time:HH:mm:ss} | {level} | {message}",
    level="DEBUG"
)

print("All imports successful")

All imports successful


Define Schemas

In [4]:
class ArticleOutput(BaseModel):
    title: str = Field(..., min_length=3, max_length=200)
    summary: str = Field(..., min_length=20, max_length=500)
    sentiment: Literal["positive", "negative", "neutral"]
    key_topics: list[str] = Field(..., min_length=1, max_length=5)
    confidence_score: float = Field(..., ge=0.0, le=1.0)

class ValidationResult(BaseModel):
    success: bool
    data: ArticleOutput | None = None
    error: str | None = None
    attempts: int
    timestamp: str

logger.info("Schemas defined successfully")

04:26:13 | INFO | Schemas defined successfully


LLM Setup

In [14]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    api_key=os.environ["GROQ_API_KEY"]
)

SYSTEM_PROMPT = """
You are a structured data extraction agent.
You MUST return ONLY valid JSON. No explanation. No markdown. No backticks.

Return EXACTLY this JSON structure with EXACTLY these field names:
{{
    "title": "string between 3 and 200 characters",
    "summary": "string between 20 and 500 characters",
    "sentiment": "positive or negative or neutral",
    "key_topics": ["array of 1 to 5 strings"],
    "confidence_score": 0.95
}}

STRICT RULES:
- Use EXACTLY these key names: title, summary, sentiment, key_topics, confidence_score
- sentiment MUST be exactly one of: positive, negative, neutral
- confidence_score MUST be a float like 0.85 not a string
- key_topics MUST be a JSON array even if only one topic
- Return NOTHING except the JSON object
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Extract structured data from this article:\n\n{article_text}")
])

chain = prompt | llm
logger.info("LLM and prompt chain initialized")

04:29:05 | INFO | LLM and prompt chain initialized


Parser + Validator

In [15]:
def parse_llm_response(response_text: str) -> dict:
    cleaned = response_text.strip()

    if "```" in cleaned:
        cleaned = re.sub(r"```(?:json)?", "", cleaned).strip()

    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group()

    parsed = json.loads(cleaned)

    field_mapping = {
        "topics": "key_topics",
        "keywords": "key_topics",
        "tags": "key_topics",
        "key_words": "key_topics",
        "main_topics": "key_topics",
        "score": "confidence_score",
        "confidence": "confidence_score",
        "confidence_level": "confidence_score",
        "headline": "title",
        "article_title": "title",
        "description": "summary",
        "text": "summary",
        "article_summary": "summary",
        "tone": "sentiment",
        "emotion": "sentiment"
    }

    normalized = {}
    for key, value in parsed.items():
        normalized_key = field_mapping.get(key.lower(), key.lower())
        normalized[normalized_key] = value

    required = ["title", "summary", "sentiment", "key_topics", "confidence_score"]
    missing = [f for f in required if f not in normalized]
    if missing:
        raise KeyError(f"Missing fields: {missing}. Got: {list(normalized.keys())}")

    sentiment_map = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "mixed": "neutral",
        "mostly positive": "positive",
        "mostly negative": "negative",
        "optimistic": "positive",
        "pessimistic": "negative"
    }
    raw_sentiment = str(normalized["sentiment"]).lower().strip()
    normalized["sentiment"] = sentiment_map.get(raw_sentiment, "neutral")

    try:
        normalized["confidence_score"] = float(normalized["confidence_score"])
    except (ValueError, TypeError):
        normalized["confidence_score"] = 0.5

    if isinstance(normalized["key_topics"], str):
        normalized["key_topics"] = [normalized["key_topics"]]

    return normalized

logger.info("Parser and normalizer defined")

04:29:08 | INFO | Parser and normalizer defined


Retry Logic + Core Agent

In [16]:
attempt_tracker = {"count": 0}

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10)
)
def call_llm_with_retry(article_text: str) -> ArticleOutput:
    attempt_tracker["count"] += 1
    logger.info(f"Attempt {attempt_tracker['count']} — Calling LLM...")

    response = chain.invoke({"article_text": article_text})
    raw = response.content
    logger.debug(f"Raw LLM response: {raw}")

    parsed = parse_llm_response(raw)
    logger.info("JSON parsed and normalized successfully")

    validated = ArticleOutput(**parsed)
    logger.info("Pydantic schema validation passed")

    return validated


def process_article(article_text: str) -> ValidationResult:
    attempt_tracker["count"] = 0

    try:
        result = call_llm_with_retry(article_text)
        return ValidationResult(
            success=True,
            data=result,
            attempts=attempt_tracker["count"],
            timestamp=datetime.now(timezone.utc).isoformat()
        )

    except ValidationError as e:
        logger.error(f"Schema validation failed: {e}")
        return ValidationResult(
            success=False,
            error=f"Validation error: {str(e)}",
            attempts=attempt_tracker["count"],
            timestamp=datetime.now(timezone.utc).isoformat()
        )

    except json.JSONDecodeError as e:
        logger.error(f"JSON parsing failed: {e}")
        return ValidationResult(
            success=False,
            error=f"JSON parse error: {str(e)}",
            attempts=attempt_tracker["count"],
            timestamp=datetime.now(timezone.utc).isoformat()
        )

    except KeyError as e:
        logger.error(f"Missing fields: {e}")
        return ValidationResult(
            success=False,
            error=f"Missing field error: {str(e)}",
            attempts=attempt_tracker["count"],
            timestamp=datetime.now(timezone.utc).isoformat()
        )

    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return ValidationResult(
            success=False,
            error=str(e),
            attempts=attempt_tracker["count"],
            timestamp=datetime.now(timezone.utc).isoformat()
        )

logger.info("Agent functions defined")

04:29:12 | INFO | Agent functions defined


Debug Cell

In [17]:
SYSTEM_PROMPT_FIXED = """
You are a structured data extraction agent.
You MUST return ONLY valid JSON. No explanation. No markdown. No backticks.

Return EXACTLY this JSON structure with EXACTLY these field names:
{{
    "title": "string between 3 and 200 characters",
    "summary": "string between 20 and 500 characters",
    "sentiment": "positive or negative or neutral",
    "key_topics": ["array of 1 to 5 strings"],
    "confidence_score": 0.95
}}

STRICT RULES:
- Use EXACTLY these key names: title, summary, sentiment, key_topics, confidence_score
- sentiment MUST be exactly one of: positive, negative, neutral
- confidence_score MUST be a float like 0.85 not a string
- key_topics MUST be a JSON array even if only one topic
- Return NOTHING except the JSON object
"""

# Re-initialize prompt and chain locally for this cell to apply the fix
prompt_fixed = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_FIXED),
    ("human", "Extract structured data from this article:\n\n{article_text}")
])

chain_fixed = prompt_fixed | llm

# Direct call — no retry, no validation, just raw output
test_article = """
Tesla reported record-breaking profits this quarter, driven by unprecedented
EV demand across European and Asian markets. CEO Elon Musk announced plans
for three new Gigafactories scheduled to open by 2026, promising to create
over 50,000 jobs globally.
"""

response = chain_fixed.invoke({"article_text": test_article})
print("RAW LLM OUTPUT:")
print(repr(response.content))
print("\n--- FORMATTED ---")
print(response.content)


RAW LLM OUTPUT:
'{\n    "title": "Tesla Reports Record Profits",\n    "summary": "Tesla reported record-breaking profits driven by high EV demand in European and Asian markets, with plans for new Gigafactories and job creation",\n    "sentiment": "positive",\n    "key_topics": ["Tesla", "Electric Vehicles", "Gigafactories"],\n    "confidence_score": 0.95\n}'

--- FORMATTED ---
{
    "title": "Tesla Reports Record Profits",
    "summary": "Tesla reported record-breaking profits driven by high EV demand in European and Asian markets, with plans for new Gigafactories and job creation",
    "sentiment": "positive",
    "key_topics": ["Tesla", "Electric Vehicles", "Gigafactories"],
    "confidence_score": 0.95
}


Test With Real Article

In [18]:
test_article = """
Tesla reported record-breaking profits this quarter, driven by unprecedented
EV demand across European and Asian markets. CEO Elon Musk announced plans
for three new Gigafactories scheduled to open by 2026, promising to create
over 50,000 jobs globally. Analysts have raised price targets across the board,
citing strong fundamentals and growing market share in the luxury EV segment.
"""

result = process_article(test_article)

print("\n========== RESULT ==========")
print(f"Success     : {result.success}")
print(f"Attempts    : {result.attempts}")
print(f"Timestamp   : {result.timestamp}")

if result.success:
    print("\n--- Extracted Data ---")
    print(f"Title          : {result.data.title}")
    print(f"Summary        : {result.data.summary}")
    print(f"Sentiment      : {result.data.sentiment}")
    print(f"Key Topics     : {result.data.key_topics}")
    print(f"Confidence     : {result.data.confidence_score}")
else:
    print(f"\nError: {result.error}")

04:29:22 | INFO | Attempt 1 — Calling LLM...
04:29:23 | DEBUG | Raw LLM response: {
    "title": "Tesla Reports Record Profits",
    "summary": "Tesla reported record-breaking profits driven by high EV demand in European and Asian markets, with plans for new Gigafactories and increased job creation, leading to raised price targets by analysts due to strong fundamentals and growing market share",
    "sentiment": "positive",
    "key_topics": ["Tesla", "Electric Vehicles", "Gigafactories"],
    "confidence_score": 0.95
}
04:29:23 | INFO | JSON parsed and normalized successfully
04:29:23 | INFO | Pydantic schema validation passed

========== RESULT ==========
Success     : True
Attempts    : 1
Timestamp   : 2026-08-01T04:29:23.303878+00:00

--- Extracted Data ---
Title          : Tesla Reports Record Profits
Summary        : Tesla reported record-breaking profits driven by high EV demand in European and Asian markets, with plans for new Gigafactories and increased job creation, leading t

Test Failure Handling

In [19]:
bad_inputs = [
    "",
    "hello",
    "random words that mean nothing at all xyz abc"
]

print("========== FAILURE HANDLING TESTS ==========\n")

for i, bad_text in enumerate(bad_inputs):
    display = bad_text[:40] + "..." if len(bad_text) > 40 else bad_text or "EMPTY"
    print(f"Test {i+1}: '{display}'")
    result = process_article(bad_text)
    print(f"Success  : {result.success}")
    print(f"Error    : {result.error}")
    print(f"Attempts : {result.attempts}")
    print("---")

========== FAILURE HANDLING TESTS ==========

Test 1: 'EMPTY'
04:29:33 | INFO | Attempt 1 — Calling LLM...
04:29:33 | DEBUG | Raw LLM response: {
    "title": "No Article Provided",
    "summary": "There is no article to summarize, please provide the text to extract data from.",
    "sentiment": "neutral",
    "key_topics": ["No Topics Found"],
    "confidence_score": 0.0
}
04:29:33 | INFO | JSON parsed and normalized successfully
04:29:33 | INFO | Pydantic schema validation passed
Success  : True
Error    : None
Attempts : 1
---
Test 2: 'hello'
04:29:33 | INFO | Attempt 1 — Calling LLM...
04:29:34 | DEBUG | Raw LLM response: {
    "title": "hello",
    "summary": "The article is empty and does not contain any information.",
    "sentiment": "neutral",
    "key_topics": ["empty article"],
    "confidence_score": 0.95
}
04:29:34 | INFO | JSON parsed and normalized successfully
04:29:34 | INFO | Pydantic schema validation passed
Success  : True
Error    : None
Attempts : 1
---
Test 3: 'r

Batch Processing

In [20]:
articles = [
    """
    Apple unveiled its latest iPhone 17 with groundbreaking AI features
    integrated directly into the chip. Pre-orders broke all previous records
    within the first 24 hours of availability across 40 countries.
    """,
    """
    Global inflation remains stubbornly high as central banks struggle
    to balance interest rate decisions. Unemployment is rising in several
    major economies, causing widespread concern among financial analysts.
    """,
    """
    The new climate agreement signed by 190 nations sets ambitious targets
    for carbon neutrality by 2040. Environmental groups cautiously welcomed
    the deal while urging faster implementation timelines.
    """
]

print("========== BATCH PROCESSING ==========\n")

results = []
for i, article in enumerate(articles):
    print(f"Processing article {i+1}...")
    result = process_article(article)
    results.append(result)

    if result.success:
        print(f"Title     : {result.data.title}")
        print(f"Sentiment : {result.data.sentiment}")
        print(f"Topics    : {result.data.key_topics}")
        print(f"Confidence: {result.data.confidence_score}")
    else:
        print(f"Failed    : {result.error}")
    print("---")

successful = sum(1 for r in results if r.success)
failed = len(results) - successful
print(f"\nBatch Summary")
print(f"Total     : {len(results)}")
print(f"Successful: {successful}")
print(f"Failed    : {failed}")

========== BATCH PROCESSING ==========

Processing article 1...
04:30:29 | INFO | Attempt 1 — Calling LLM...
04:30:29 | DEBUG | Raw LLM response: {
    "title": "iPhone 17 Release",
    "summary": "Apple unveiled its latest iPhone 17 with groundbreaking AI features integrated directly into the chip, breaking pre-order records across 40 countries within the first 24 hours.",
    "sentiment": "positive",
    "key_topics": ["iPhone 17", "AI features", "Apple"],
    "confidence_score": 0.95
}
04:30:29 | INFO | JSON parsed and normalized successfully
04:30:29 | INFO | Pydantic schema validation passed
Title     : iPhone 17 Release
Sentiment : positive
Topics    : ['iPhone 17', 'AI features', 'Apple']
Confidence: 0.95
---
Processing article 2...
04:30:29 | INFO | Attempt 1 — Calling LLM...
04:30:30 | DEBUG | Raw LLM response: {
    "title": "Global Inflation Remains High",
    "summary": "Global inflation remains high, central banks struggle with interest rate decisions, and unemployment ris

Project Summary

In [21]:
print("========== SCHEMASHIELD SUMMARY ==========\n")
print("Project     : SchemaShield — Structured Output Agent")
print("Author      : K Murali Krishna")
print("Model       : Groq LLaMA3-8b-8192")
print("Validation  : Pydantic v2")
print("Retry Logic : Tenacity exponential backoff (max 3 attempts)")
print("Logging     : Loguru structured logs")
print("\nKey Capabilities:")
print("  ✓ Schema enforcement on every LLM response")
print("  ✓ Field name normalization for model inconsistencies")
print("  ✓ Sentiment value normalization")
print("  ✓ Automatic retry with exponential backoff")
print("  ✓ Graceful failure handling with full error logging")
print("  ✓ Batch processing support")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Never trust LLM output — always verify")
print("  ✓ Retry transient failures, log permanent ones")
print("  ✓ Structured logging for observability")
print("  ✓ Schema-first design")

========== SCHEMASHIELD SUMMARY ==========

Project     : SchemaShield — Structured Output Agent
Author      : K Murali Krishna
Model       : Groq LLaMA3-8b-8192
Validation  : Pydantic v2
Retry Logic : Tenacity exponential backoff (max 3 attempts)
Logging     : Loguru structured logs

Key Capabilities:
  ✓ Schema enforcement on every LLM response
  ✓ Field name normalization for model inconsistencies
  ✓ Sentiment value normalization
  ✓ Automatic retry with exponential backoff
  ✓ Graceful failure handling with full error logging
  ✓ Batch processing support

Production Concepts Demonstrated:
  ✓ Never trust LLM output — always verify
  ✓ Retry transient failures, log permanent ones
  ✓ Structured logging for observability
  ✓ Schema-first design
